## Read in TROPESS

In [3]:
import sys
import os
import numpy as np
import xarray as xr
import pandas as pd

import datetime
import matplotlib as mpl
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')


In [4]:

from functools import partial
def _preprocess_pres(x, ):
    x = x.rename({'latitude':'lat','longitude':'lon','x_h2o':'H2O'})
    x = x.assign_coords({'time':x.time})
    x = x.assign_coords({'lat':x.lat})
    x = x.assign_coords({'lon':x.lon})
    x = x.assign_coords({'level':x.pressure[0,:].values})                               
    x = x.drop_vars(["pressure"])
    x = x.assign_coords({"target": pd.MultiIndex.from_arrays(
        [x.time.values, x.lat.values, x.lon.values],
        names=["time", "lon", "lat"])}).unstack("target")
    x = x.transpose("time", "level", "lon", "lat")
    
    #print('**level**', x.level.data)
    #print('*****x',x)

    lon_meshgrid, lat_meshgrid = np.meshgrid(x.lon.values, x.lat.values, indexing='ij')
    
    x_out = xr.Dataset(
        data_vars=dict(
        ratio=(["time","level","x","y"], x.x.data),
        H2O=(["time","level","x","y"], x.H2O.data),
        
    ),
        coords=dict(
        time=(["time"], x.time.data),
        level=(["level"], x.level.data),
        lon=(["x","y"], lon_meshgrid),
        lat=(["x","y"], lat_meshgrid),
    ),
    )
    
    #print(x_out)
    return x_out
partial_func_pres = partial(_preprocess_pres)

arr = os.listdir('/Users/ellendyer/Documents/GitHub/Isotopes_F4R/forward/central_test/')
print(arr)
for I,F in enumerate(arr):
    print(F)
    x = xr.open_dataset('/Users/ellendyer/Documents/GitHub/Isotopes_F4R/forward/central_test/'+F,
                        engine="netcdf4",
                        drop_variables=['x_h2o_col_p_error','x_h2o_col_p','dd_004','dd_col_p',
                                        'dd_col_p_error','datetime_utc','datetime_utc_dim', 
                                        'altitude','level','target_idx','target_id','year_fraction'])
    
    x = x.rename({'latitude':'lat','longitude':'lon','x_h2o':'H2O'})
    #x = x.assign_coords({'time':x.time})
    #x = x.assign_coords({'lat':x.lat})
    #x = x.assign_coords({'lon':x.lon})
    #x = x.assign_coords({'level':x.pressure[0,:].values})
    #x = x.assign_coords({'level':[1.0142621e+03, 1.0000000e+03, 9.0851398e+02, 
    #                              8.2540198e+02, 7.4989301e+02, 6.8129102e+02, 
    #                              6.1896600e+02, 5.1089801e+02, 4.2169800e+02, 
    #                              3.4806900e+02, 2.8729800e+02, 2.3713699e+02, 
    #                              1.7782899e+02, 1.3335201e+02, 7.4989601e+01, 
    #                              2.8729900e+01, 1.0000000e-01]})                               
    #x = x.drop_vars(["pressure"])
    #x = x.assign_coords({"target": pd.MultiIndex.from_arrays(
    #    [x.time.values, x.lat.values, x.lon.values],
    #    names=["time", "lon", "lat"])}).unstack("target")
    #x = x.transpose("time", "level", "lon", "lat") 
    #x = x.sortby("level", ascending=False)
    print(x)
    
    if I == 0:
     ds = x
    else:
     ds = xr.concat([ds,x],dim='time')
    x.close()

print(ds)
print(ds.ratio.values)

['TROPESS_AIRS-Aqua_L2_Summary_HDO_20210214_MUSES_R1p11_FS_F0p6.SUB.nc4', 'TROPESS_AIRS-Aqua_L2_Summary_HDO_20210211_MUSES_R1p11_FS_F0p6.SUB.nc4', 'TROPESS_AIRS-Aqua_L2_Summary_HDO_20210209_MUSES_R1p11_FS_F0p6.SUB.nc4', 'TROPESS_AIRS-Aqua_L2_Summary_HDO_20210321_MUSES_R1p11_FS_F0p6.SUB.nc4', 'TROPESS_AIRS-Aqua_L2_Summary_HDO_20210210_MUSES_R1p11_FS_F0p6.SUB.nc4', 'TROPESS_AIRS-Aqua_L2_Summary_HDO_20210215_MUSES_R1p11_FS_F0p6.SUB.nc4', 'TROPESS_AIRS-Aqua_L2_Summary_HDO_20210213_MUSES_R1p11_FS_F0p6.SUB.nc4']
TROPESS_AIRS-Aqua_L2_Summary_HDO_20210214_MUSES_R1p11_FS_F0p6.SUB.nc4
<xarray.Dataset> Size: 268kB
Dimensions:   (target: 1198, level: 17)
Coordinates:
  * target    (target) float32 5kB 2.29e+03 2.292e+03 ... 4.514e+03 4.515e+03
    lon       (target) float32 5kB ...
    lat       (target) float32 5kB ...
Dimensions without coordinates: level
Data variables:
    time      (target) datetime64[ns] 10kB ...
    pressure  (target, level) float32 81kB ...
    x         (target, level) fl

ValueError: time already exists as coordinate or variable name.

In [ ]:
print(np.nanmax(ds.ratio.sel(level = 825,method='nearest').values))
print(np.nanmin(ds.ratio.sel(level = 825,method='nearest').values))

In [ ]:
# calculate deltaD
R = ds.ratio
# use this RSMOW for molecules of water
Rvsmow = 3.11*10**(-4)
dD = (R/Rvsmow - 1)*1000.
ds['deltaD'] = dD

In [ ]:
print(np.nanmax(ds.deltaD.sel(level = 825,method='nearest').values))
print(np.nanmin(ds.deltaD.sel(level = 825,method='nearest').values))

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature
ax = plt.axes(projection=ccrs.PlateCarree())
ax.coastlines()
ax.add_feature(cartopy.feature.OCEAN)
ds['deltaD'].mean('time').sel(level = 825,method='nearest').plot(ax=ax,transform=ccrs.PlateCarree(),
                                       add_colorbar=True,
                                       #vmin=0,vmax=2,
                                       alpha=1,
                                       cmap=plt.cm.RdBu,
                                       extend="both")

ax.set_extent([-20, 20, -20, 60])
#ax.set_extent([7, 32, -15, 12])
gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True,
                  linewidth=1, color='black', alpha=0.5, linestyle='dotted')
gl.top_labels = False
gl.right_labels = False
#ax.set_title()
#plt.savefig()
plt.show()
plt.clf()

In [ ]:
cb_tropess = ds.where((ds.lat>-5)&(ds.lat<5)&(ds.lon>10)&(ds.lon<28), drop=True)

print(cb_tropess)

In [ ]:

cb_tropess.to_netcdf(path='/Users/ellendyer/Documents/GitHub/Isotopes_F4R/iso_prepped/tropess_airs_cb.nc',mode='w',format='NETCDF4',engine='netcdf4')